In [1]:
from geostat_isoscapes_tools import geostat_utils as gutils, plot_utils as putils, sisal_utils as sutils, utils
import matplotlib.pyplot as plt 
import seaborn as sns
import pandas as pd 
import json
import os 
import numpy as np
import json
import plotly.io as pio
pio.renderers.default = "vscode"
sns.set_style('dark')

In this notebook, we compare variograms of itrace and sisal data. In particular, we select itrace data with the union of circles centered around sisal points at each time slice.
Hence, we developped a function that automatically defines the mask, iterate over itrace time slices to compute their variograms, then aggregate them and save the results. 

This workflow is can be run using the script ```variogram_parameters_scan```, where all parameters can be set and looped on.
The results will automatically be saved in output folders (see logs when running).

> The original detailed workflow used before implementing the parameters scan is available in the section below

This notebook can still be useful to visualize the trend fitting results (metrics)

### Detailed workflow 

[ now superseded by script binvariogram_parameters_scan ]

In [ ]:
params = {'trend': 'multiple_linear_latabs_latReLU_ele_D',
          'trend_before_mask': True,
          'maxlag': 1.2e7,
          'nlags': 20,
          'centers': None,
          'tolerance' : 22.5,
          'mask radius [km]': 1000,
          'res [months]': 12*200,
          'Itrace simulation spec' : {'time':'20kyrBP',},
          'regions':None,
          'const_coords':True
          }
data_cols = {'lat':'lat',
             'lon':'lon',
             'quantity':'d18Op'}

fp_itrace = f"../output/variograms/itrace/sim{params['Itrace simulation spec']['time']}/res{params['res [months]']}/maxlag{str(int(params['maxlag']))}nlags{params['nlags']}/trend_{params['trend']}/"
if not os.path.exists(fp_itrace) :
    os.makedirs(fp_itrace)

fp_sisal = f"../output/variograms/sisal/res{params['res [months]']}/maxlag{str(int(params['maxlag']))}nlags{params['nlags']}/trend_{params['trend']}/"
if not os.path.exists(fp_sisal) :
    os.makedirs(fp_sisal)

verbose =  False

In [ ]:
# Load sisal data 
sisal_df = gutils.get_sisal_data_for_kriging(res=int(params['res [months]']/12),regions=params['countries'],verbose=verbose)
# load itrace data
# itrace_data = gutils.get_preprocessed_itrace_data(res=params['res [months]'],P=True,format='xr',verbose=False,
#                                             sim_prefix = 'b.e13.Bi1850C5.f19_g16',
#                                             sim_suffix ='100001-199912', #12:'800001-899912', #16:'400001-499912', #20 :'100001-199912',
#                                             sim_forcings='ice_ghg_orb_wtr',
#                                             sim_kyr= int(params['Itrace simulation spec']['time'][:2]),
#                                             sim_num='01',
#                                             sim_model='clm2.h0',
#                                             ) # xr


#### **Mask iTrace data around SISAL points**

In [ ]:
sisal_df = gutils.get_sisal_data_for_kriging(res=int(params['res [months]']/12),countries=params['countries'],verbose=verbose)

itrace_df = gutils.get_preprocessed_itrace_data(res=params['res [months]'],P=True,format='df',verbose=False,
                                            sim_prefix = 'b.e13.Bi1850C5.f19_g16',
                                            sim_suffix ='100001-199912', #12:'800001-899912', #16:'400001-499912', #20 :'100001-199912',
                                            sim_forcings='ice_ghg_orb_wtr',
                                            sim_kyr= 20,
                                            sim_num='01',
                                            sim_model='clm2.h0',
                                            ) # pd df

In [ ]:
itrace_masked_df,_ = utils.mask_union_of_circles_around_pts(itrace_df,sisal_df,params['mask radius [km]'],verbose=True) # type:ignore

In [ ]:
fig = putils.plot_global_map(data=itrace_df.groupby(['time']).mean().reset_index(),
                    quantity_col=data_cols['quantity'],
                    title = 'masked itrace data',
                    quantity=data_cols['quantity'],
                    unit='per mil (VSMOW)',
                    proj=False,
                    symbol='square',
                    size=3,
                    lat_col=data_cols['lat'],
                    lon_col=data_cols['lon'])

# fig.write_html("map.html", auto_open=True)
fig.show()

#### **Compute variograms**

#### iTrace variograms

In [ ]:
for azimuth in [None] : #None,0,45,90,135
    print(f'computation dir={azimuth}')
    params['direction']=azimuth
    gutils.iterate_and_aggregate_variograms(data_ds = itrace_data, #type:ignore
                                        fp=fp_itrace,
                                        config_dict=params,
                                        data_cols=data_cols,
                                        mask_pts = sisal_df, # type:ignore
                                        verbose = verbose
                                       )

In [ ]:
dict_dfs= {
           'i_all': {'df': pd.read_csv(fp_itrace+'vario_df.csv')},
        #    'i_0': {'df': pd.read_csv(fp_itrace+'d0vario_df.csv')},
        #    'i_45': {'df': pd.read_csv(fp_itrace+'d45vario_df.csv')},
        #    'i_90': {'df': pd.read_csv(fp_itrace+'d90vario_df.csv')},
        #    'i_135': {'df': pd.read_csv(fp_itrace+'d135vario_df.csv')}
        }

model_name = 'spherical'

for key in dict_dfs.keys():
    dict_dfs[key]['params'],dict_dfs[key]['fct_fitted'],dict_dfs[key]['pcov'] = gutils.fit_variogram_model(bins=dict_dfs[key]['df']['lag'], # type:ignore
                                                                                gammas=dict_dfs[key]['df']['gamma'],
                                                                                model_name=model_name,
                                                                                pair_counts=dict_dfs[key]['df']['count']
                                                                                ) 

In [ ]:
fig,ax = putils.plot_variogram_from_bins_and_gamma(centers=dict_dfs['i_all']['df']['lag'],
                                          gamma=dict_dfs['i_all']['df']['gamma'],
                                          time='iTrace dataset - 11.7 to 11 ky BP - weighted average',
                                          counts=dict_dfs['i_all']['df']['count'],
                                          min_pairs=20,
                                          plot_model=True,
                                          model_name=model_name,
                                          model_fct=dict_dfs['i_all']['fct_fitted'],
                                          model_params=dict_dfs['i_all']['params'],
                                          figsize=(10,5),
                                          save_name=fp_itrace+'fig.png'
                                          ) # type:ignore
plt.show()
plt.close('all')

In [ ]:
list_dir = [0,45,90,135]
ranges=[]
fig,axes = plt.subplots(2,2,figsize=(20,10))
for i in range(2):
    for j in range(2):
        key = f'i_{list_dir[j+2*i]}'
        axes[i,j] = putils.plot_variogram_from_bins_and_gamma(centers=dict_dfs[key]['df']['lag'],
                                                            gamma=dict_dfs[key]['df']['gamma'],
                                                            time='iTrace dataset - 11.7 to 11 ky BP - weighted average',
                                                            counts=dict_dfs[key]['df']['count'],
                                                            min_pairs=20,
                                                            plot_model=True,
                                                            model_name=model_name,
                                                            model_fct=dict_dfs[key]['fct_fitted'],
                                                            model_params=dict_dfs[key]['params'],
                                                            figsize=(10,5),
                                                            ax_ = axes[i,j]
                                                            #     save_name='../output/variograms/fig_global_variogram_and_model_chen.png'
                                                            ) # type:ignore
        axes[i,j].set_title(f'dir={list_dir[j+2*i]}')
        ranges.append(dict_dfs[key]['params']['range'])
        
plt.show()
plt.close('all')

for dir, r in zip(list_dir, ranges):
    print(f'Range in direction {dir}°: {r :.2e} m')
anisotropy_ratio = max(ranges) / min(ranges)
print(f'Anisotropy Ratio: {anisotropy_ratio}')
max_range_index = ranges.index(max(ranges))
print(f'Anisotropy Direction: {list_dir[max_range_index]}°')

#### SISAL variograms

In [ ]:
sisal_df_valid = sisal_df.loc[(sisal_df['binned_age']>=19000)&(sisal_df['binned_age']<20000),['binned_age','lat','lon','site_id','d18Op_VSMOW_exactconv']].copy()

In [ ]:
putils.plot_global_map(data=sisal_df_valid[sisal_df_valid['binned_age']==19000],
                       lon_col='lon',
                       lat_col='lat',   
                       quantity_col='d18Op_VSMOW_exactconv',
                       unit='per mil (VSMOW)',
                       proj=True,
                       title='SISAL 19-20 kyr BP'
                       )

In [ ]:
params['direction']=None
params['const_coords']=False
gutils.iterate_and_aggregate_variograms(data = sisal_df_valid.rename(columns={'d18Op_VSMOW_exactconv':'d18Op','binned_age':'time'}), #type:ignore
                                        fp=fp_sisal,
                                        config_dict=params,
                                        data_cols=data_cols,
                                        mask_pts = None, # type:ignore
                                        verbose = False
                                       )

### Visualize metrics of trend fitting 

In [37]:
import numpy as np
import json

# put the path of the results file directory here 
fp = '../output/variograms/itrace/sim12kyrBP/res2400/maxlag3000000nlags20/trend_multiple_linear_lat_ele_D/no_mask/countries_Australia/'#subregions_NothernEuropeSouthernEuropeWesternEurope/'

with open(fp+'trend_metricsNone.json', 'r') as f: # 'None' refers to the (un)directionality of the variogram ('azimuth' in parameters scan)
    results_dict = json.load(f)

list_df =[]
metrics_dict = {}
for i,t in enumerate(list(results_dict.keys())):
    df = gutils.multiple_linear_result_dict_to_df(results_dict[t])
    df['time']=float(t)
    df['r2']=results_dict[t]['r2']
    df['r2adj']=results_dict[t]['adj_r2']
    df['mae']=results_dict[t]['mae']
    list_df.append(df)

result_df = pd.concat(list_df).reset_index(drop=True)

print('Mean MAE :',np.mean(result_df['mae']))
print('Mean r2 :',np.mean(result_df['r2']))
print('Mean r2adj :',np.mean(result_df['r2adj']))

Mean MAE : 0.6164429635280773
Mean r2 : 0.6357828694454604
Mean r2adj : 0.6296443762716739


In [38]:
result_df[['name','beta','std_error','p_value','vif','partial r2','r2','r2adj','mae']]

,name,beta,std_error,p_value,vif,partial r2,r2,r2adj,mae
0,intercept,-2.145407e+00,2.243996e-01,9.713340e-18,NaN,NaN,0.633433,0.627255,0.591706
1,lat,1.243605e-01,7.686430e-03,8.466178e-37,2.296709,0.539074,0.633433,0.627255,0.591706
2,ele,-1.617535e-03,2.997367e-04,2.145905e-07,2.601571,0.059974,0.633433,0.627255,0.591706
3,D,-7.669805e-07,2.424787e-07,1.835831e-03,2.811550,0.020604,0.633433,0.627255,0.591706
4,intercept,-2.065504e+00,2.349729e-01,1.242983e-15,NaN,NaN,0.623113,0.616761,0.625901
5,lat,1.283638e-01,8.048600e-03,3.863627e-36,2.296709,0.538562,0.623113,0.616761,0.625901
6,ele,-1.610217e-03,3.138597e-04,7.510749e-07,2.601571,0.055730,0.623113,0.616761,0.625901
7,D,-7.454696e-07,2.539039e-07,3.763525e-03,2.811550,0.018252,0.623113,0.616761,0.625901
8,intercept,-2.034667e+00,2.307669e-01,1.053837e-15,NaN,NaN,0.633294,0.627114,0.614383
9,lat,1.288004e-01,7.904533e-03,3.970425e-37,2.296709,0.546991,0.633294,0.627114,0.614383


In [12]:
result_df.loc[result_df['name']=='|lat|','partial r2']

1     0.023550
5     0.018929
9     0.018844
13    0.013575
17    0.013641
Name: partial r2, dtype: float64